# Call Center Transcript Analysis

Analyze call center transcripts using OpenAI to extract customer name, intent, and resolution.

In [1]:
import os
import csv
import json
from openai import OpenAI

## Setup

In [2]:
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
INPUT_FILE = "callcenter.csv"

## Prompt and response format

In [3]:
SYSTEM_PROMPT = """You analyze call center transcripts. Extract these fields:
- CustomerName
- CustomerIntent
- Resolution

Each value MUST be under 30 characters. Be very concise."""

RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "call_analysis",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "CustomerName":    {"type": "string"},
                "CustomerIntent":  {"type": "string"},
                "Resolution":      {"type": "string"},
            },
            "required": [
                "CustomerName", "CustomerIntent", "Resolution",
            ],
            "additionalProperties": False,
        },
    },
}

## Analyze function

In [4]:
def analyze(text):
    resp = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": text},
        ],
        temperature=0,
        response_format=RESPONSE_FORMAT,
    )
    return json.loads(resp.choices[0].message.content)

## Read input and process transcripts

In [5]:
with open(INPUT_FILE, encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

results = []
for row in rows:
    print(f"Processing {row['RecordID']}...")
    extracted = analyze(row["TextScript"])
    results.append({
        "RecordID": row["RecordID"],
        "CustomerID": row["CustomerID"],
        "AgentName": row["AgentName"],
        "CustomerName": extracted["CustomerName"],
        "CustomerIntent": extracted["CustomerIntent"],
        "Resolution": extracted["Resolution"],
    })

results

Processing Rec-01...
Processing Rec-02...
Processing Rec-03...
Processing Rec-04...
Processing Rec-05...


[{'RecordID': 'Rec-01',
  'CustomerID': 'Cust-01',
  'AgentName': 'Agent-Nina',
  'CustomerName': 'Somchai Prasert',
  'CustomerIntent': 'Report fraud transaction',
  'Resolution': 'Card blocked, dispute filed'},
 {'RecordID': 'Rec-02',
  'CustomerID': 'Cust-02',
  'AgentName': 'Agent-David',
  'CustomerName': 'Suda Chaiyasit',
  'CustomerIntent': 'Request fee waiver',
  'Resolution': 'Annual fee waived'},
 {'RecordID': 'Rec-03',
  'CustomerID': 'Cust-03',
  'AgentName': 'Agent-Amy',
  'CustomerName': 'Niran Kittipong',
  'CustomerIntent': 'Request payment assistance',
  'Resolution': '6-month installment plan offered'},
 {'RecordID': 'Rec-04',
  'CustomerID': 'Cust-04',
  'AgentName': 'Agent-Sophia',
  'CustomerName': 'Kanya Rattanakul',
  'CustomerIntent': 'Check card decline reason',
  'Resolution': 'Card unlocked, alerts set'},
 {'RecordID': 'Rec-05',
  'CustomerID': 'Cust-05',
  'AgentName': 'Agent-James',
  'CustomerName': 'Pimchanok Leelawat',
  'CustomerIntent': 'Add travel not